# 01 — Audit du corpus et de la provenance

**Objectif** : vérifier les documents, les dates, les entités, les sources et les faits avant de juger le retrieval.

**Critère de passage** : chaque chunk référence un document existant ; chaque fait possède une entité, une période et une valeur exploitable.

In [1]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()
from app.store.artifacts import store

store.corpus_version

'foyer-public-demo-2026-08-21.1'

In [2]:
document_rows = [
    {
        'id': document.id,
        'titre': document.title,
        'année': document.year,
        'entité': document.entity,
        'statut': document.status,
        'source': document.source_url,
    }
    for document in store.documents
]
display_table(document_rows)

,id,titre,année,entité,statut,source
0,foyer_financial_information_2025,Informations financières — Groupe Foyer,2025,Groupe Foyer,reviewed,https://groupe.foyer.lu/fr/foyer/informations-...
1,foyer_annual_report_2025,Rapport annuel 2025 — présentation web,2025,Groupe Foyer,reviewed,https://groupe.foyer.lu/fr/rapport-annuel
2,table_extraction_demo,Annexe synthétique — contrôle de cellules,2025,Entité de démonstration,synthetic,https://groupe.foyer.lu/fr/documents
3,foyer_group_qrt_2025,QRT public 2025 - Groupe Foyer,2025,Groupe Foyer,reviewed,https://www.foyer.lu/fr/mydoc/WebSites-Documen...
4,foyer_assurances_qrt_2025,QRT Public 2025 - Foyer Assurances,2025,Foyer Assurances S.A.,reviewed,https://www.foyer.lu/fr/mydoc/WebSites-Documen...
5,foyer_global_health_qrt_2025,QRT Public 2025 - Foyer Global Health,2025,Foyer Global Health S.A.,reviewed,https://www.foyer.lu/fr/mydoc/WebSites-Documen...
6,foyer_governance_charter_2025,Foyer Group Governance Charter,2025,Groupe Foyer,reviewed,https://www.foyer.lu/fr/mydoc/WebSites-Documen...
7,foyer_sustainability_statement,Foyer Sustainability Statement,2025,Groupe Foyer,reviewed,https://www.foyer.lu/fr/mydoc/12625
8,foyer_sfcr_2025,Foyer Group Solvency and Financial Condition R...,2025,Groupe Foyer,reviewed,https://www.foyer.lu/fr/mydoc/WebSites-Documen...


In [3]:
fact_rows = []
for chunk in store.chunks:
    for fact in chunk.facts:
        fact_rows.append({
            'champ': fact.field_id,
            'valeur': fact.formatted_value,
            'type': fact.value_type,
            'entité': fact.entity,
            'période': fact.period,
            'document': chunk.document_id,
            'section': ' › '.join(chunk.locator.section_path),
        })
display_table(fact_rows)

,champ,valeur,type,entité,période,document,section
0,group_equity,"1 544,0 M€",currency,Groupe Foyer,2025-12-31,foyer_financial_information_2025,Le Groupe Foyer › Chiffres clés
1,non_life_market_share,40 %,percentage,Groupe Foyer,2025,foyer_financial_information_2025,Le Groupe Foyer › Position de marché
2,business_areas,"3 domaines : non-vie, vie et gestion patrimoniale",text,Groupe Foyer,2025,foyer_financial_information_2025,Le Groupe Foyer › Domaines d'activité
3,insured_households,> 166 000,integer,Foyer au Luxembourg,2025,foyer_annual_report_2025,Assurance au Luxembourg › Chiffres clés
4,myfoyer_satisfaction,"93,5 %",percentage,Foyer au Luxembourg,2025,foyer_annual_report_2025,Assurance au Luxembourg › Expérience digitale
5,claims_satisfaction,93 %,percentage,Foyer au Luxembourg,2025,foyer_annual_report_2025,Assurance au Luxembourg › Gestion des sinistres
6,active_countries,> 100 pays,integer,Global Health,2025,foyer_annual_report_2025,Global Health › Présence internationale
7,earned_premiums,129 M€,currency,Global Health,2025,foyer_annual_report_2025,Global Health › Indicateurs
8,employee_count,111 employés,integer,Global Health,2025,foyer_annual_report_2025,Global Health › Indicateurs
9,eligible_own_funds_scr,2 407 647 kEUR,currency,Groupe Foyer,2025-12-31,foyer_group_qrt_2025,S.23.01.22 › Fonds propres


In [4]:
document_ids = {document.id for document in store.documents}
invalid_chunks = [
    chunk.id for chunk in store.chunks
    if chunk.document_id not in document_ids
    or chunk.locator.document_id != chunk.document_id
    or not chunk.locator.source_url.startswith('https://')
    or chunk.locator.page < 1
]
synthetic_documents = [document.id for document in store.documents if document.status == 'synthetic']
assert not invalid_chunks, f'Provenance incomplète : {invalid_chunks}'
{
    'documents': len(document_ids),
    'chunks': len(store.chunks),
    'facts': len(fact_rows),
    'invalid_chunks': invalid_chunks,
    'synthetic_documents_explicitly_labeled': synthetic_documents,
}

{'documents': 9,
 'chunks': 1918,
 'facts': 18,
 'invalid_chunks': [],
 'synthetic_documents_explicitly_labeled': ['table_extraction_demo']}

### Point de contrôle

Une bonne métrique de retrieval ne compense jamais une période erronée, une source introuvable ou une valeur synthétique présentée comme réelle. Corriger le corpus avant d'optimiser les modèles.